In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [17]:
# other required dependencies
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
import string

# 1. Load data

In [3]:
# loading the dataset
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [4]:
# overview of data
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [5]:
test.head(3)

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...


# 2. Quick sanity checks

In [6]:
train.isna().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

In [7]:
test.isna().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
dtype: int64

# 3. Check the hypothesis: does option length predict the correct answer?

In [18]:
OPTION_COLS = ['A', 'B', 'C', 'D', 'E']

def content_word_len(text):
    text = str(text).lower()

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Split into words
    words = text.split()

    # Remove stop words
    words = [w for w in words if w not in ENGLISH_STOP_WORDS]

    return len(words)

len_matrix = train[OPTION_COLS].applymap(content_word_len).values  # shape (n_rows, 5)

# rank options by length, longest = rank 0
len_rank_idx = np.argsort(-len_matrix, axis=1)  # descending
len_rank_letters = np.array(OPTION_COLS)[len_rank_idx]

# where does the correct answer fall in the length ranking?
true_answers = train['answer'].values
rank_position = np.array([
    list(row).index(ans) for row, ans in zip(len_rank_letters, true_answers)
])

print("Distribution of correct answer's rank-by-length (0 = longest option):")
print(pd.Series(rank_position).value_counts(normalize=True).sort_index().round(3))

Distribution of correct answer's rank-by-length (0 = longest option):
0    0.397
1    0.240
2    0.168
3    0.106
4    0.088
Name: proportion, dtype: float64


/tmp/ipykernel_58/2386502655.py:17: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  len_matrix = train[OPTION_COLS].applymap(content_word_len).values  # shape (n_rows, 5)


Distribution of correct answer's rank-by-length (0 = longest option):
0    0.194
1    0.194
2    0.232
3    0.202
4    0.178
Name: proportion, dtype: float64


/tmp/ipykernel_58/3623292559.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  char_matrix = train[OPTION_COLS].applymap(lambda x: len(str(x))).values


# 4. Apply the same concept on test dataset

In [19]:
len_matrix = test[OPTION_COLS].applymap(content_word_len).values  # shape (500, 5)

# rank options by length, longest = rank 0
len_rank_idx = np.argsort(-len_matrix, axis=1)  # descending
len_rank_letters = np.array(OPTION_COLS)[len_rank_idx]

# where does the correct answer fall in the length ranking?
true_answers = train['answer'].values
rank_position = np.array([
    list(row).index(ans) for row, ans in zip(len_rank_letters, true_answers)
])

/tmp/ipykernel_58/1428084157.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  len_matrix = test[OPTION_COLS].applymap(content_word_len).values  # shape (500, 5)


# 5. Make the submission

In [21]:
submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": [" ".join(row[:3]) for row in len_rank_letters]
})

submission.to_csv("submission.csv", index=False)
submission.head()

,ID,Prediction
0,1,B C D
1,2,A B C
2,3,E B C
3,4,C E B
4,5,A B C


# Hold other code cells

# 3. Build the TF-IDF vectorizer

In [8]:
# def combined_text(df):
#     return df[['prompt'] + OPTION_COLS].astype(str).agg(' '.join, axis=1)

# all_text = pd.concat([combined_text(train), combined_text(test)], ignore_index=True)

# vectorizer = TfidfVectorizer(stop_words="english")
# vectorizer.fit(all_text)

# print("Vocabulary size:", len(vectorizer.get_feature_names_out()))

Vocabulary size: 2762


# 4. Similarity-based ranking function

In [9]:
# def rank_options(df, vectorizer, option_cols=OPTION_COLS):
#     """Returns an array of shape (n_rows, 5) with option letters ranked
#     from most to least similar to the prompt, for every row in df."""
#     prompt_vecs = vectorizer.transform(df['prompt'].astype(str))

#     sims = {}
#     for opt in option_cols:
#         opt_vecs = vectorizer.transform(df[opt].astype(str))
#         # row-wise cosine similarity without building the full pairwise matrix
#         sims[opt] = np.array(prompt_vecs.multiply(opt_vecs).sum(axis=1)).flatten()

#     sim_matrix = np.column_stack([sims[opt] for opt in option_cols])

#     # sort descending (most similar first)
#     ranked_idx = np.argsort(sim_matrix, axis=1)[:, ::-1]
#     ranked_letters = np.array(option_cols)[ranked_idx]

#     return ranked_letters

# 5. Sanity-check on the training set

In [10]:
# def map_at_3(true_answers, ranked_letters):
#     scores = []
#     for ans, ranks in zip(true_answers, ranked_letters):
#         ranks = list(ranks[:3])
#         if ans in ranks:
#             scores.append(1 / (ranks.index(ans) + 1))
#         else:
#             scores.append(0)
#     return np.mean(scores)

# train_ranked = rank_options(train, vectorizer)
# train_map3 = map_at_3(train['answer'], train_ranked)
# print(f"Train MAP@3 (TF-IDF baseline): {train_map3:.4f}")

Train MAP@3 (TF-IDF baseline): 0.2579


# 6. Generate predictions on the test set

In [11]:
# test_ranked = rank_options(test, vectorizer)
# test_top3 = test_ranked[:, :3]

# # join into a single space-separated string, e.g. "C A B"
# predictions = [' '.join(row) for row in test_top3]

# test_ranked[:5], test_top3[:5]

(array([['E', 'D', 'C', 'B', 'A'],
        ['E', 'D', 'C', 'B', 'A'],
        ['A', 'D', 'C', 'B', 'E'],
        ['A', 'E', 'C', 'D', 'B'],
        ['A', 'C', 'B', 'E', 'D']], dtype='<U1'),
 array([['E', 'D', 'C'],
        ['E', 'D', 'C'],
        ['A', 'D', 'C'],
        ['A', 'E', 'C'],
        ['A', 'C', 'B']], dtype='<U1'))

# 7. Build the submission file

In [12]:
# submission = pd.DataFrame({
#     'ID': test['id'],
#     'Prediction': predictions
# })

# submission.head()

,ID,Prediction
0,1,E D C
1,2,E D C
2,3,A D C
3,4,A E C
4,5,A C B


In [13]:
# submission.to_csv('submission.csv', index=False)
# print("Saved submission.csv with shape:", submission.shape)

Saved submission.csv with shape: (500, 2)
